# Visualize training slices

Reads one axial slice from `<dataset>/<case>.zip` via the SSL index (`image_path`, `z`).
Each volume is 128 slices in `[0, 1]`; anatomy should sit near the middle of the stack.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lib.zarr_loader import read_volume_slice

STORE = ROOT / "data" / "processed" / "slices_zarr"
N_SHOW, SEED, SPLIT = 16, 42, "train"

table = next(p for p in (STORE / "slices.parquet", STORE / "slices.csv") if p.exists())
slices = pd.read_parquet(table) if table.suffix == ".parquet" else pd.read_csv(table)
if SPLIT:
    slices = slices[slices["split"].astype(str) == SPLIT]
print(f"{table.name}: {len(slices)} slices under {STORE}")
sample = slices.sample(n=min(N_SHOW, len(slices)), random_state=SEED)

ncols = 4
nrows = int(np.ceil(len(sample) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 3.4 * nrows))
for ax, (_, row) in zip(np.atleast_1d(axes).ravel(), sample.iterrows()):
    path = STORE / str(row["image_path"])
    img = np.asarray(read_volume_slice(path, int(row["z"])), dtype=np.float32).squeeze()
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{row['modality']}/{row['dataset']} z={int(row['z'])}\n{row['volume_id']}", fontsize=7)
    ax.axis("off")
for ax in np.atleast_1d(axes).ravel()[len(sample):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
row = slices.sample(n=1, random_state=SEED + 1).iloc[0]
path = STORE / str(row["image_path"])
stack = np.stack([read_volume_slice(path, z).squeeze() for z in range(128)])
tissue = (stack > 0.05).mean(axis=(1, 2))
print(row[["volume_id", "dataset", "modality", "split", "image_path"]].to_string())
print(
    f"tissue_frac mid±8={tissue[56:72].mean():.3f}  "
    f"z0-15={tissue[:16].mean():.3f}  z112-127={tissue[-16:].mean():.3f}"
)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, z in zip(axes.ravel(), np.linspace(0, 127, 16, dtype=int)):
    ax.imshow(stack[z], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"z={z}", fontsize=8)
    ax.axis("off")
fig.suptitle(str(row["volume_id"]))
fig.tight_layout()
plt.show()